##### 1. Experiment with hyperparameter tuning for a GPT model and document the effects on model performance.
● Choose a set of hyperparameters to tune (e.g., learning rate, batch size, numberof epochs).

● Conduct experiments to observe the impact of different hyperparameter values on model performance.

● Summarize the results in a detailed report (1000-1200 words), including plots or tables to visualize performance changes.
##### 2. Compare and contrast different variants of GPT models in a written report, including their applications and capabilities.
● Research different variants of GPT models (e.g., GPT-2, GPT-3, GPT-Neo, GPT-J).

● Write a comprehensive report (1500-2000 words) comparing their architectures, training data, and performance.

● Discuss the specific applications and capabilities of each variant, providing realworld examples where possible.

In [5]:
import torch 
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling,)
from datasets import load_dataset 
import math 
import matplotlib.pyplot as plt 
import pandas as pd

In [6]:
# checking gpu availabilty 
device = "cuda" if torch.cuda.is_available() else "cpu" 
print (f"Using {device} device")

Using cuda device


##### Loading the dataset and preparing the dataset

In [7]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print("Dataset structure:")
print(dataset)
print(f"\nTraining examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nSample text:")
print(dataset["train"][100]["text"][:300])

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Training examples: 36718
Validation examples: 3760

Sample text:
 96 ammunition packing boxes 



##### tokenizing the dataset

In [9]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# gpt 2 doesnt have padding toekn by default so i set one 
tokenizer.pad_token = tokenizer.eos_token 
BLOCK_SIZE = 128

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=BLOCK_SIZE)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset = tokenized_dataset.filter(lambda x: any(t != tokenizer.pad_token_id for t in x["input_ids"]))

print(f"Tokenized training examples: {len(tokenized_dataset['train'])}")
print(f"Tokenized validation examples: {len(tokenized_dataset['validation'])}")
    

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Tokenized training examples: 23767
Tokenized validation examples: 2461
